In [147]:
# Initialize Otter
import otter
grader = otter.Notebook("hw8.ipynb")

# CPSC 330 - Applied Machine Learning

## Homework 8: Introduction to Computer vision and Time Series

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [148]:
from hashlib import sha1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>

_Points:_ 2

<!-- END QUESTION -->

<br><br>

## Exercise 1: time series prediction

In this exercise we'll be looking at a [dataset of avocado prices](https://www.kaggle.com/neuromusic/avocado-prices). You should start by downloading the dataset and storing it under the `data` folder. We will be forcasting average avocado price for the next week. 

In [149]:
df = pd.read_csv("data/avocado.csv", parse_dates=["Date"], index_col=0)
df.head()

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


In [150]:
df.shape

(18249, 13)

In [151]:
df["Date"].min()

Timestamp('2015-01-04 00:00:00')

In [152]:
df["Date"].max()

Timestamp('2018-03-25 00:00:00')

It looks like the data ranges from the start of 2015 to March 2018 (~2 years ago), for a total of 3.25 years or so. Let's split the data so that we have a 6 months of test data.

In [153]:
split_date = '20170925'
df_train = df[df["Date"] <= split_date]
df_test  = df[df["Date"] >  split_date]

In [154]:
assert len(df_train) + len(df_test) == len(df)

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 How many time series? 
rubric={points:4}

In the [Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package) dataset from lecture demo, we had different measurements for each Location. 

We want you to consider this for the avocado prices dataset. For which categorical feature(s), if any, do we have separate measurements? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 4

The avocado dataset includes different measurements for the Region, as well as the Type of the avocado (conventional or organic). We can see that there are avocadoes with different regions and types on the same date in the dataset. Each region is able to have the two sub-types of avocado.

In [155]:
df["region"].unique()

array(['Albany', 'Atlanta', 'BaltimoreWashington', 'Boise', 'Boston',
       'BuffaloRochester', 'California', 'Charlotte', 'Chicago',
       'CincinnatiDayton', 'Columbus', 'DallasFtWorth', 'Denver',
       'Detroit', 'GrandRapids', 'GreatLakes', 'HarrisburgScranton',
       'HartfordSpringfield', 'Houston', 'Indianapolis', 'Jacksonville',
       'LasVegas', 'LosAngeles', 'Louisville', 'MiamiFtLauderdale',
       'Midsouth', 'Nashville', 'NewOrleansMobile', 'NewYork',
       'Northeast', 'NorthernNewEngland', 'Orlando', 'Philadelphia',
       'PhoenixTucson', 'Pittsburgh', 'Plains', 'Portland',
       'RaleighGreensboro', 'RichmondNorfolk', 'Roanoke', 'Sacramento',
       'SanDiego', 'SanFrancisco', 'Seattle', 'SouthCarolina',
       'SouthCentral', 'Southeast', 'Spokane', 'StLouis', 'Syracuse',
       'Tampa', 'TotalUS', 'West', 'WestTexNewMexico'], dtype=object)

In [156]:
df["type"].unique()

array(['conventional', 'organic'], dtype=object)

In [157]:
df.sort_values(by=["Date"]).head()

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
51,2015-01-04,1.75,27365.89,9307.34,3844.81,615.28,13598.46,13061.10,537.36,0.0,organic,2015,Southeast
51,2015-01-04,1.49,17723.17,1189.35,15628.27,0.00,905.55,905.55,0.00,0.0,organic,2015,Chicago
51,2015-01-04,1.68,2896.72,161.68,206.96,0.00,2528.08,2528.08,0.00,0.0,organic,2015,HarrisburgScranton
51,2015-01-04,1.52,54956.80,3013.04,35456.88,1561.70,14925.18,11264.80,3660.38,0.0,conventional,2015,Pittsburgh
51,2015-01-04,1.64,1505.12,1.27,1129.50,0.00,374.35,186.67,187.68,0.0,organic,2015,Boise


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Equally spaced measurements? 
rubric={points:4}

In the Rain in Australia dataset, the measurements were generally equally spaced but with some exceptions. How about with this dataset? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 4

Almost all of the regions (in combination with the type) have a pretty equal spacing of 7 days, with 168 entries each except for WestTexNewMexico, which has a count of 163 for 7 days, one entry 14 days after, and one entry 21 days after (indicators of missing data). When examining the categories individually we can see that they lead to uneven data because for the region based sorting, they are comparing both organic and conventional from the same day, leading to a double count. For type, the overlap is huge because it's comparing avocados of the respective type for every single region. Because of this, we will need to consider the two measurements together (type as a sub measurement to region) in order to have properly spaced data.

In [158]:
# taken from lecture 20
def plot_time_spacing_distribution_region(df, region=""):
    """
    Plots the distribution of time spacing for a given region.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with columns 'region' and 'Date'.
        region (str): The region (e.g., location) to analyze.
    """
    # Ensure 'Date' is in datetime format
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter data for the given region
    region_data = df[df['region'] == region]
    
    if region_data.empty:
        print(f"No data available for region: {region}")
        return
    
    # Calculate time differences
    time_diffs = region_data['Date'].sort_values().diff().dropna()
    
    # Count the frequency of each time difference
    value_counts = time_diffs.value_counts().sort_index()
    
    # Display value counts
    print(f"Time spacing counts for {region}:\n{value_counts}\n")

In [159]:
for region in df["region"].unique():
    plot_time_spacing_distribution_region(df, region)

Time spacing counts for Albany:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for Atlanta:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for BaltimoreWashington:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for Boise:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for Boston:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for BuffaloRochester:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for California:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for Charlotte:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for Chicago:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for CincinnatiDayton:
Date
0 days    169
7 days    168
Name: count, dtype: int64

Time spacing counts for Columbus:
Date


In [160]:
# taken from lecture 20
def plot_time_spacing_distribution_type(df, type=""):
    """
    Plots the distribution of time spacing for a given type.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with columns 'type' and 'Date'.
        type (str): The type (e.g., organic/conventional) to analyze.
    """
    # Ensure 'Date' is in datetime format
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter data for the given region
    region_data = df[df['type'] == type]
    
    if region_data.empty:
        print(f"No data available for type: {type}")
        return
    
    # Calculate time differences
    time_diffs = region_data['Date'].sort_values().diff().dropna()
    
    # Count the frequency of each time difference
    value_counts = time_diffs.value_counts().sort_index()
    
    # Display value counts
    print(f"Time spacing counts for {type}:\n{value_counts}\n")

In [161]:
for type in df["type"].unique():
    plot_time_spacing_distribution_type(df, type)

Time spacing counts for conventional:
Date
0 days    8957
7 days     168
Name: count, dtype: int64

Time spacing counts for organic:
Date
0 days    8954
7 days     168
Name: count, dtype: int64



In [162]:
# taken from lecture 20
def plot_time_spacing_distribution_both(df, region="", type=""):
    """
    Plots the distribution of time spacing for a given type.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with columns 'type', 'region' and 'Date'.
        type (str): The type (e.g., organic/conventional) to analyze.
        region (str): The region to analyze.
    """
    # Ensure 'Date' is in datetime format
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter data for the given region
    region_data = df[(df['type'] == type) & (df["region"] == region)]
    
    if region_data.empty:
        print(f"No data available for type/region: {type} {region}")
        return
    
    # Calculate time differences
    time_diffs = region_data['Date'].sort_values().diff().dropna()
    
    # Count the frequency of each time difference
    value_counts = time_diffs.value_counts().sort_index()
    
    # Display value counts
    print(f"Time spacing counts for {type} {region}:\n{value_counts}\n")

In [163]:
for region in df["region"].unique():
    for type in df["type"].unique():
        plot_time_spacing_distribution_both(df, region, type)

Time spacing counts for conventional Albany:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for organic Albany:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for conventional Atlanta:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for organic Atlanta:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for conventional BaltimoreWashington:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for organic BaltimoreWashington:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for conventional Boise:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for organic Boise:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for conventional Boston:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for organic Boston:
Date
7 days    168
Name: count, dtype: int64

Time spacing counts for conventional BuffaloRochester:
Date
7 days    168
Name: count, dt

In [164]:
...

Ellipsis

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Interpreting regions 
rubric={points:4}

In the Rain in Australia dataset, each location was a different place in Australia. For this dataset, look at the names of the regions. Do you think the regions are also all distinct, or are there overlapping regions? Justify your answer by referencing the data.

<div class="alert alert-warning">

Solution_1.3
    
</div>

_Points:_ 4

No, there are overlapping regions in this dataset. This is seen by the biggest region "TotalUS" as well as other smaller overlapping regions like "West", "Northeast", "Southeast","GreatLakes", "Plains", "SouthCentral", and "Midsouth". This is seen when you take the total volume of all the regions except for TotalUS, we actually exceed the total volume found in TotalUS, indicating overlap.

In [165]:
df['region'].unique()

array(['Albany', 'Atlanta', 'BaltimoreWashington', 'Boise', 'Boston',
       'BuffaloRochester', 'California', 'Charlotte', 'Chicago',
       'CincinnatiDayton', 'Columbus', 'DallasFtWorth', 'Denver',
       'Detroit', 'GrandRapids', 'GreatLakes', 'HarrisburgScranton',
       'HartfordSpringfield', 'Houston', 'Indianapolis', 'Jacksonville',
       'LasVegas', 'LosAngeles', 'Louisville', 'MiamiFtLauderdale',
       'Midsouth', 'Nashville', 'NewOrleansMobile', 'NewYork',
       'Northeast', 'NorthernNewEngland', 'Orlando', 'Philadelphia',
       'PhoenixTucson', 'Pittsburgh', 'Plains', 'Portland',
       'RaleighGreensboro', 'RichmondNorfolk', 'Roanoke', 'Sacramento',
       'SanDiego', 'SanFrancisco', 'Seattle', 'SouthCarolina',
       'SouthCentral', 'Southeast', 'Spokane', 'StLouis', 'Syracuse',
       'Tampa', 'TotalUS', 'West', 'WestTexNewMexico'], dtype=object)

In [166]:
df[df['region']=='TotalUS'].head()

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-12-27,0.95,27297983.67,9626901.09,10197890.05,1184340.09,6288852.44,4850404.09,1252626.31,185822.04,conventional,2015,TotalUS
1,2015-12-20,0.98,25083647.17,8710021.76,9329861.85,1201020.01,5842743.55,4618389.66,1025048.77,199305.12,conventional,2015,TotalUS
2,2015-12-13,0.93,28041335.38,9855053.66,10805838.91,1016163.17,6364279.64,4964462.13,1371440.28,28377.23,conventional,2015,TotalUS
3,2015-12-06,0.89,28800396.57,9405464.36,12160838.62,931830.63,6302262.96,5005077.36,1233956.21,63229.39,conventional,2015,TotalUS
4,2015-11-29,0.99,22617999.38,8094803.56,9003178.41,731008.41,4789009.00,3901953.04,856560.34,30495.62,conventional,2015,TotalUS


In [167]:
total_vol = 0

for region in df['region'].unique():
    if region == 'TotalUS':
        continue
    temp= df[(df['region']==region) & (df['Date']=="2015-12-27")]
    total_vol += temp['Total Volume'].sum()

total_vol

np.float64(45959232.59000001)

<!-- END QUESTION -->

<br><br>

We will use the entire dataset despite any location-based weirdness uncovered in the previous part.

We will be trying to forecast the avocado price. The function below is adapted from the lecture.

In [168]:
def create_lag_feature(
    df: pd.DataFrame,
    orig_feature: str,
    lag: int,
    groupby: list[str],
    new_feature_name: str | None = None,
    clip: bool = False,
) -> pd.DataFrame:
    """
    Create a lagged (or ahead) version of a feature, optionally per group.

    Assumes df is already sorted by time within each group and has unique indices.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset.
    orig_feature : str
        Name of the column to lag.
    lag : int
        The lag:
          - negative → values from the past (t-1, t-2, ...)
          - positive → values from the future (t+1, t+2, ...)
    groupby : list of str
        Column(s) to group by if df contains multiple time series.
    new_feature_name : str, optional
        Name of the new column. If None, a name is generated automatically.
    clip : bool, default False
        If True, drop rows where the new feature is NaN.

    Returns
    -------
    pd.DataFrame
        A new dataframe with the additional column added.
    """
    if lag == 0:
        raise ValueError("lag cannot be 0 (no shift). Use the original feature instead.")

    # Default name if not provided
    if new_feature_name is None:
        if lag < 0:
            new_feature_name = f"{orig_feature}_lag{abs(lag)}"
        else:
            new_feature_name = f"{orig_feature}_ahead{lag}"

    df = df.copy()

    # Map your convention (negative=past, positive=future) to pandas shift
    # pandas: shift(+k) → past, shift(-k) → future
    periods = abs(lag) if lag < 0 else -lag

    df[new_feature_name] = (
        df.groupby(groupby, sort=False)[orig_feature]
          .shift(periods)
    )

    if clip:
        df = df.dropna(subset=[new_feature_name])

    return df


We first sort our dataframe properly:

In [169]:
df_sort = df.sort_values(by=["region", "type", "Date"]).reset_index(drop=True)
df_sort

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico
18247,2018-03-18,1.56,15896.38,2055.35,1499.55,0.00,12341.48,12114.81,226.67,0.0,organic,2018,WestTexNewMexico


We then call `create_lag_feature`. This creates a new column in the dataset `AveragePriceNextWeek`, which is the following week's `AveragePrice`. We have set `clip=True` which means it will remove rows where the target would be missing.

In [170]:
df_hastarget = create_lag_feature(df_sort, "AveragePrice", +1, ["region", "type"], "AveragePriceNextWeek", clip=True)
df_hastarget

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region,AveragePriceNextWeek
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany,1.24
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany,1.17
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany,1.06
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany,0.99
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18243,2018-02-18,1.56,17597.12,1892.05,1928.36,0.00,13776.71,13553.53,223.18,0.0,organic,2018,WestTexNewMexico,1.57
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico,1.54
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico,1.56
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico,1.56


Our goal is to predict `AveragePriceNextWeek`. 

Let's split the data:

In [171]:
df_train = df_hastarget[df_hastarget["Date"] <= split_date]
df_test  = df_hastarget[df_hastarget["Date"] >  split_date]

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 `AveragePrice` baseline 
rubric={points}

Soon we will want to build some models to forecast the average avocado price a week in advance. Before we start with any ML though, let's try a baseline. Previously we used `DummyClassifier` or `DummyRegressor` as a baseline. This time, we'll do something else as a baseline: we'll assume the price stays the same from this week to next week. So, we'll set our prediction of "AveragePriceNextWeek" exactly equal to "AveragePrice", assuming no change. That is kind of like saying, "If it's raining today then I'm guessing it will be raining tomorrow". This simplistic approach will not get a great score but it's a good starting point for reference. If our model does worse that this, it must not be very good. 

Using this baseline approach, what $R^2$ do you get on the train and test data?

<div class="alert alert-warning">

Solution_1.4
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [172]:
df_baseline = df_train['AveragePrice']

In [173]:
train_r2 = r2_score(df_train['AveragePriceNextWeek'], df_baseline)
train_r2

0.8285800937261841

In [174]:
df_baseline = df_test['AveragePrice']

In [175]:
test_r2 = r2_score(df_test['AveragePriceNextWeek'], df_baseline)
test_r2

0.7631780188583048

In [176]:
assert not train_r2 is None, "Are you using the correct variable name?"
assert not test_r2 is None, "Are you using the correct variable name?"
assert sha1(str(round(train_r2, 3)).encode('utf8')).hexdigest() == 'b1136fe2a8918904393ab6f40bfb3f38eac5fc39', "Your training score is not correct. Are you using the right features?"
assert sha1(str(round(test_r2, 3)).encode('utf8')).hexdigest() == 'cc24d9a9b567b491a56b42f7adc582f2eefa5907', "Your test score is not correct. Are you using the right features?"

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Forecasting average avocado price
rubric={points:10}

Now that the baseline is done, let's build some models to forecast the average avocado price a week later. Experiment with a few approachs for encoding the date. Justify the decisions you make. Which approach worked best? Report your test score and briefly discuss your results.

Benchmark: you should be able to achieve $R^2$ of at least 0.79 on the test set. I got to 0.80, but not beyond that. Let me know if you do better!

Note: because we only have 2 splits here, we need to be a bit wary of overfitting on the test set. Try not to test on it a ridiculous number of times. If you are interested in some proper ways of dealing with this, see for example sklearn's [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), which is like cross-validation for time series data.

<div class="alert alert-warning">

Solution_1.5
    
</div>

_Points:_ 10

First, we will try simple Posix encoding with month, week, and sin and cos of week.

In [177]:
# convert to POSIX time
df_train.loc[:, "Date_posix"] = df_train["Date"].astype("int64") // 10**9
df_test.loc[:, "Date_posix"] = df_test["Date"].astype("int64") // 10**9

#add month, week
df_train.loc[:,"month"] = df_train["Date"].dt.month
df_train.loc[:,"week"] = df_train["Date"].dt.isocalendar().week
df_test.loc[:,"month"] = df_test["Date"].dt.month
df_test.loc[:,"week"] = df_test["Date"].dt.isocalendar().week

# add sin and cos of week to capture differences in seasons
df_train.loc[:,"week_sin"] = np.sin(2 * np.pi * df_train["Date"].dt.isocalendar().week / 52)
df_train.loc[:,"week_cos"] = np.cos(2 * np.pi * df_train["Date"].dt.isocalendar().week / 52)
df_test.loc[:,"week_sin"] = np.sin(2 * np.pi * df_test["Date"].dt.isocalendar().week / 52)
df_test.loc[:,"week_cos"] = np.cos(2 * np.pi * df_test["Date"].dt.isocalendar().week / 52)

X_train = df_train.drop(columns=["Date", "AveragePriceNextWeek"])
X_test = df_test.drop(columns=["Date", "AveragePriceNextWeek"])
y_train = df_train["AveragePriceNextWeek"]
y_test = df_test["AveragePriceNextWeek"]

C:\Users\justh\AppData\Local\Temp\ipykernel_3860\734163597.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train.loc[:, "Date_posix"] = df_train["Date"].astype("int64") // 10**9
C:\Users\justh\AppData\Local\Temp\ipykernel_3860\734163597.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, "Date_posix"] = df_test["Date"].astype("int64") // 10**9
C:\Users\justh\AppData\Local\Temp\ipykernel_3860\734163597.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

In [ ]:
# adapted from lecture 20

regressor = RandomForestRegressor(n_estimators=50, random_state=0)
simp = SimpleImputer()
X_train_rfr = simp.fit_transform(X_train.drop(columns = ["type", "region"]))
X_test_rfr = simp.transform(X_test.drop(columns = ["type", "region"]))
regressor.fit(X_train_rfr, y_train)
print("Train-set R^2: {:.2f}".format(regressor.score(X_train_rfr, y_train)))
print("Test-set R^2: {:.2f}".format(regressor.score(X_test_rfr, y_test)))

Train-set R^2: 0.98
Test-set R^2: 0.75


Here we tried using a basic Random Forest Regressor without doing any preprocessing on the columns. Next, we will try to perform preprocessing on the categorical features. Next, we will use the appropriate preprocessing technique on each column. We will use standard scaling, ordinal encoding, and one hot encoding.

In [179]:
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

print(df_train.head())
numeric_features = ["Date_posix", "AveragePrice", "Total Volume", "4046", "4225", "4770", "Total Bags", "Small Bags", "Large Bags", "XLarge Bags", "year", "month", "week", "week_sin", "week_cos"]

categorical_features = ["region"]

ordinal_features = ["type"]

numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler()
)

categorical_pipeline = make_pipeline(
    OneHotEncoder(handle_unknown="ignore")
)

ordinal_pipeline = make_pipeline(
    OrdinalEncoder(categories=[["conventional", "organic"]])
)

preprocessor = make_column_transformer(
    (numeric_pipeline, numeric_features),
    (categorical_pipeline, categorical_features),
    (ordinal_pipeline, ordinal_features),
)

preprocessor

        Date  AveragePrice  Total Volume     4046      4225    4770  \
0 2015-01-04          1.22      40873.28  2819.50  28287.42   49.90   
1 2015-01-11          1.24      41195.08  1002.85  31640.34  127.12   
2 2015-01-18          1.17      44511.28   914.14  31540.32  135.77   
3 2015-01-25          1.06      45147.50   941.38  33196.16  164.14   
4 2015-02-01          0.99      70873.60  1353.90  60017.20  179.32   

   Total Bags  Small Bags  Large Bags  XLarge Bags          type  year  \
0     9716.46     9186.93      529.53          0.0  conventional  2015   
1     8424.77     8036.04      388.73          0.0  conventional  2015   
2    11921.05    11651.09      269.96          0.0  conventional  2015   
3    10845.82    10103.35      742.47          0.0  conventional  2015   
4     9323.18     9170.82      152.36          0.0  conventional  2015   

   region  AveragePriceNextWeek  Date_posix  month  week  week_sin  week_cos  
0  Albany                  1.24  1420329600      

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

We will now try a random forest regressor with this new preprocessor.

In [ ]:
rfr_pipe = make_pipeline(
    preprocessor,
    RandomForestRegressor(n_estimators=50, random_state=0)
)
rfr_pipe.fit(X_train, y_train)
print("Train-set R^2: {:.2f}".format(rfr_pipe.score(X_train, y_train)))
print(r2_score(df_train['AveragePriceNextWeek'], rfr_pipe.predict(X_train)))
print("Test-set R^2: {:.2f}".format(rfr_pipe.score(X_test, y_test)))
print(r2_score(df_test['AveragePriceNextWeek'], rfr_pipe.predict(X_test)))

Train-set R^2: 0.98
0.9802963515887672
Test-set R^2: 0.75
0.7519575900811827


We see very little difference in scores here. So we will try adding more lag features to give the model more to work off of.

In [181]:
df_train = create_lag_feature(df_train, "AveragePrice", -1, ["region", "type"], "lag_1")
df_train = create_lag_feature(df_train, "AveragePrice", -2, ["region", "type"], "lag_2")
df_train = create_lag_feature(df_train, "AveragePrice", -3, ["region", "type"], "lag_3")

df_test = create_lag_feature(df_test, "AveragePrice", -1, ["region", "type"], "lag_1")
df_test = create_lag_feature(df_test, "AveragePrice", -2, ["region", "type"], "lag_2")
df_test = create_lag_feature(df_test, "AveragePrice", -3, ["region", "type"], "lag_3")

X_train = df_train.drop(columns=["Date", "AveragePriceNextWeek"])
X_test = df_test.drop(columns=["Date", "AveragePriceNextWeek"])
y_train = df_train["AveragePriceNextWeek"]
y_test = df_test["AveragePriceNextWeek"]

numeric_features = ["Date_posix", "AveragePrice", "Total Volume", "4046", "4225", "4770", "Total Bags", "Small Bags", "Large Bags", "XLarge Bags", "year", "month", "week", "week_sin", "week_cos", "lag_1", "lag_2", "lag_3"]

preprocessor = make_column_transformer(
    (numeric_pipeline, numeric_features),
    (categorical_pipeline, categorical_features),
    (ordinal_pipeline, ordinal_features),
)

In [182]:
rfr_pipe = make_pipeline(
    preprocessor,
    RandomForestRegressor(n_estimators=50, random_state=0)
)
rfr_pipe.fit(X_train, y_train)
print("Train-set R^2: {:.2f}".format(rfr_pipe.score(X_train, y_train)))
print(r2_score(df_train['AveragePriceNextWeek'], rfr_pipe.predict(X_train)))
print("Test-set R^2: {:.2f}".format(rfr_pipe.score(X_test, y_test)))
print(r2_score(df_test['AveragePriceNextWeek'], rfr_pipe.predict(X_test)))

Train-set R^2: 0.98
0.9815580140201539
Test-set R^2: 0.78
0.775648213128356


Now, we will add some more metrics to hopefully get more accurate. Additionally, we will start changing the max_depth hyperparameter because these models so far look to be heavily overfitting. We will also use hyperparamter optimization by using grid search

In [183]:
df_train["rolling_mean_4"] = (df_train.groupby(["region", "type"])["AveragePrice"].transform(lambda x: x.rolling(4).mean()))
df_train["rolling_std_4"] = (df_train.groupby(["region", "type"])["AveragePrice"].transform(lambda x: x.rolling(4).std()))
df_test["rolling_mean_4"] = (df_test.groupby(["region", "type"])["AveragePrice"].transform(lambda x: x.rolling(4).mean()))
df_test["rolling_std_4"] = (df_test.groupby(["region", "type"])["AveragePrice"].transform(lambda x: x.rolling(4).std()))

X_train = df_train.drop(columns=["Date", "AveragePriceNextWeek"])
X_test = df_test.drop(columns=["Date", "AveragePriceNextWeek"])
y_train = df_train["AveragePriceNextWeek"]
y_test = df_test["AveragePriceNextWeek"]

numeric_features = ["Date_posix", "AveragePrice", "Total Volume", "4046", "4225", "4770", "Total Bags", "Small Bags", "Large Bags", "XLarge Bags", "year", "month", "week", "week_sin", "week_cos", "lag_1", "lag_2", "lag_3", "rolling_mean_4", "rolling_std_4"]

preprocessor = make_column_transformer(
    (numeric_pipeline, numeric_features),
    (categorical_pipeline, categorical_features),
    (ordinal_pipeline, ordinal_features),
)

In [184]:
rfr_pipe = make_pipeline(
    preprocessor,
    RandomForestRegressor(n_estimators=10, max_depth=5, random_state=0)
)
rfr_pipe.fit(X_train, y_train)
print("Train-set R^2: {:.2f}".format(rfr_pipe.score(X_train, y_train)))
print(r2_score(df_train['AveragePriceNextWeek'], rfr_pipe.predict(X_train)))
print("Test-set R^2: {:.2f}".format(rfr_pipe.score(X_test, y_test)))
print(r2_score(df_test['AveragePriceNextWeek'], rfr_pipe.predict(X_test)))


Train-set R^2: 0.86
0.8600229189141869
Test-set R^2: 0.80
0.7952963513980702


Now we will try to use ridge and see if we can get similar performance. We will remove the POSIX column as this column will confuse the ridge model, due to its inability to capture seasonality.

In [185]:
numeric_features = [
    "AveragePrice", "Total Volume", "4046", "4225", "4770",
    "Total Bags", "Small Bags", "Large Bags", "XLarge Bags",
    "year", "month", "week", "week_sin", "week_cos",
    "lag_1", "lag_2", "lag_3",
    "rolling_mean_4", "rolling_std_4"
]

drop_features = [
    "Date_posix"
]

preprocessor = make_column_transformer(
    (numeric_pipeline, numeric_features),
    (categorical_pipeline, categorical_features),
    (ordinal_pipeline, ordinal_features),
    ("drop", drop_features),
)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

In [186]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

param_grid = {
    "ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

ridge_pipe = make_pipeline(
    preprocessor,
    Ridge()
)

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=tscv,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...e', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'ridge__alpha': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed

In [187]:
best_model = grid_search.best_estimator_

print("Best alpha:", grid_search.best_params_)
print("Train R^2: {:.2f}".format(best_model.score(X_train, y_train)))
print(r2_score(df_train['AveragePriceNextWeek'], best_model.predict(X_train)))
print("Test R^2: {:.2f}".format(best_model.score(X_test, y_test)))
print(r2_score(df_test['AveragePriceNextWeek'], best_model.predict(X_test)))

Best alpha: {'ridge__alpha': 100.0}
Train R^2: 0.86
0.8580356176135843
Test R^2: 0.80
0.8016865209617665


This ridge model performs similary to the Random Forest Regressor while being computationally more light. Here we show the coefficients for each feature.

In [188]:
ridge_model = best_model.named_steps["ridge"]

feature_names = best_model.named_steps["columntransformer"].get_feature_names_out()

coefficients = ridge_model.coef_

coef_df = list(zip(feature_names, coefficients))

coef_df_sorted = sorted(coef_df, key=lambda x: abs(x[1]), reverse=True)

for feature, coef in coef_df_sorted:
    print(f"{feature}: {coef:.4f}")

pipeline-1__AveragePrice: 0.2314
pipeline-1__lag_1: 0.0535
pipeline-3__type: 0.0531
pipeline-1__lag_3: 0.0385
pipeline-2__region_SanFrancisco: 0.0359
pipeline-2__region_HartfordSpringfield: 0.0346
pipeline-2__region_Houston: -0.0307
pipeline-1__lag_2: 0.0292
pipeline-2__region_NewYork: 0.0268
pipeline-2__region_DallasFtWorth: -0.0267
pipeline-2__region_SouthCentral: -0.0230
pipeline-2__region_Denver: -0.0220
pipeline-2__region_Sacramento: 0.0209
pipeline-2__region_Charlotte: 0.0204
pipeline-2__region_CincinnatiDayton: -0.0195
pipeline-2__region_Philadelphia: 0.0194
pipeline-2__region_Nashville: -0.0193
pipeline-1__week_cos: -0.0180
pipeline-1__week_sin: -0.0165
pipeline-2__region_Northeast: 0.0165
pipeline-2__region_PhoenixTucson: -0.0149
pipeline-2__region_Roanoke: -0.0146
pipeline-2__region_RaleighGreensboro: 0.0145
pipeline-2__region_Columbus: -0.0143
pipeline-2__region_Detroit: -0.0138
pipeline-2__region_Chicago: 0.0138
pipeline-2__region_LosAngeles: -0.0136
pipeline-2__region_Alba

From this we can see that Average price, lag_1, and type impact the answer the most.

In [189]:
...

Ellipsis

In [190]:
...

Ellipsis

In [191]:
...

Ellipsis

In [192]:
...

Ellipsis

In [193]:
...

Ellipsis

In [194]:
...

Ellipsis

In [195]:
...

Ellipsis

In [196]:
...

Ellipsis

In [197]:
...

Ellipsis

In [198]:
...

Ellipsis

In [199]:
...

Ellipsis

In [200]:
...

Ellipsis

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Short answer questions

<!-- BEGIN QUESTION -->

### 2.1 Time series

rubric={points:6}

The following questions pertain to Lecture 20 on time series data:

1. Sometimes a time series has missing time points or, worse, time points that are unequally spaced in general. Give an example of a real world situation where the time series data would have unequally spaced time points.
2. In class we discussed two approaches to using temporal information: encoding the date as one or more features, and creating lagged versions of features. Which of these (one/other/both/neither) two approaches would struggle with unequally spaced time points? Briefly justify your answer.
3. When studying time series modeling, we explored several ways to encode date information as a feature for the citibike dataset. When we used time of day as a numeric feature, the Ridge model was not able to capture the periodic pattern. Why? How did we tackle this problem? Briefly explain.

<div class="alert alert-warning">

Solution_2.1
    
</div>

_Points:_ 6

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Computer vision 
rubric={points:6}

The following questions pertain to the lecture on multiclass classification and introduction to computer vision. 

1. How many parameters (coefficients and intercepts) will `sklearn`’s `LogisticRegression()` model learn for a four-class classification problem, assuming that you have 10 features? Briefly explain your answer.
2. In Lecture 19, we briefly discussed how neural networks are sort of like `sklearn`'s pipelines, in the sense that they involve multiple sequential transformations of the data, finally resulting in the prediction. Why was this property useful when it came to transfer learning?
3. Imagine that you have a small dataset with ~1000 images containing pictures and names of 50 different Computer Science faculty members from UBC. Your goal is to develop a reasonably accurate multi-class classification model for this task. Describe which model/technique you would use and briefly justify your choice in one to three sentences.

<div class="alert alert-warning">

Solution_2.2
    
</div>

_Points:_ 6

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)